In [78]:
import pandas as pd
import numpy as np

# Set seed for reproducibility of hidden signals
np.random.seed(42)
num_rows = 10000

# Generate Timestamps over a 48-hour window in 2026
timestamps = pd.to_datetime('2026-05-18 00:00:00') + pd.to_timedelta(np.random.randint(0, 172800, num_rows), unit='s')

# Create Base Telemetry
senders = ['it-support@microsoft-security.net', 'payroll@workday-benefits.co', 'billing@invoice-vendor.com', 
           'ceo@company-updates.co', 'newsletter@aws.amazon.com', 'internal-hr@company.com']
sender_pool = np.random.choice(senders, num_rows, p=[0.05, 0.05, 0.10, 0.05, 0.45, 0.30])

recipients = [f'u{i}' for i in range(1, 201)]
recipient_pool = np.random.choice(recipients, num_rows)

llm_prob = np.random.uniform(0.0, 0.4, num_rows)
links = np.random.poisson(0.5, num_rows)

telemetry_df = pd.DataFrame({
    'email_id': [f'm_{i}' for i in range(num_rows)],
    'sender_address': sender_pool,
    'recipient_id': recipient_pool,
    'timestamp': timestamps,
    'llm_generated_prob': llm_prob,
    'contained_links': links,
    'reported_malicious': 0
})

# INJECT HIDDEN ADVERSARIAL SIGNATURES (The Attack Wave)
# Attack Vector 1: Highly personalized LLM-spearphishing targeting Executive Suite
exec_users = [f'u{i}' for i in range(1, 11)] # High-value targets
attack_idx_1 = telemetry_df[(telemetry_df['recipient_id'].isin(exec_users)) & 
                            (telemetry_df['sender_address'] == 'ceo@company-updates.co')].index
telemetry_df.loc[attack_idx_1, 'llm_generated_prob'] = np.random.uniform(0.92, 0.99, len(attack_idx_1))
telemetry_df.loc[attack_idx_1, 'reported_malicious'] = 1

# Attack Vector 2: High-velocity malicious spray window
spray_condition = (telemetry_df['sender_address'] == 'it-support@microsoft-security.net') & \
                  (telemetry_df['timestamp'].dt.hour.isin([9, 10, 11]))
attack_idx_2 = telemetry_df[spray_condition].index
telemetry_df.loc[attack_idx_2, 'contained_links'] = telemetry_df.loc[attack_idx_2, 'contained_links'] + np.random.randint(2, 5, len(attack_idx_2))
telemetry_df.loc[attack_idx_2, 'reported_malicious'] = 1

# Save as Partitioned/Large Format Parquet Simulation
telemetry_df.to_parquet('large_inbound_telemetry.parquet', index=False)

# 2. DOMAIN INTEL REGISTRY
registry_data = {
    'domain_name': ['company.com', 'microsoft-security.net', 'invoice-vendor.com', 
                    'company-updates.co', 'aws.amazon.com', 'workday-benefits.co'],
    'domain_age_days': [4500, 1, 120, 2, 7800, 4],
    'historical_reputation_score': [0.99, 0.02, 0.85, 0.04, 0.99, 0.07]
}
pd.DataFrame(registry_data).to_parquet('large_domain_intel.parquet', index=False)

# 3. USER RISK PROFILES
user_data = {
    'user_id': [f'u{i}' for i in range(1, 201)],
    'department': np.random.choice(['Executive Suite', 'Human Resources', 'Engineering', 'Sales'], 200, p=[0.05, 0.15, 0.40, 0.40]),
    'security_clearance_level': [3 if i <= 10 else np.random.randint(1, 3) for i in range(1, 201)],
    'historical_fail_rate': [np.random.uniform(0.35, 0.60) if i <= 10 else np.random.uniform(0.0, 0.25) for i in range(1, 201)]
}
pd.DataFrame(user_data).to_parquet('large_user_intelligence.parquet', index=False)

print(f"Mothership 3.0 Online. Telemetry matrix initialized with {num_rows} transactions.")

Mothership 3.0 Online. Telemetry matrix initialized with 10000 transactions.


## The following parquet files are included from the Mothership
- large_inbound_telemetry.parquet
- large_domain_intel.parquet
- large_user_intelligence.parquet

In [79]:
# Import libraries
import pandas as pd

# Phase 1 
# Read parquet files - partially loading columns saves memory as less dataframes take up space from the beginning - chunking
lit = pd.read_parquet('large_inbound_telemetry.parquet', columns=['email_id', 'sender_address', 'recipient_id', 'timestamp', 'reported_malicious', 'llm_generated_prob'])
ldi = pd.read_parquet('large_domain_intel.parquet')
lui = pd.read_parquet('large_user_intelligence.parquet')

# Parquet files are already columnized and works using a vectorized scheme - which is already quite fast compared to csv files. 
# This makes it unnecessary to use chunking on parquet files

# Phase 2
# lit.info(), ldi.info(), lui.info()

# 1. Parse out the sender domain lookup key.
lit['domain_name'] = lit['sender_address'].str.extract(r'^(?:.*)@(.+)$')

# 2. Left-join all data tables seamlessly.
df = pd.merge(
    (pd.merge(
        lit,
        ldi,
        on='domain_name',
        how='left'
    )),
    lui,
    left_on='recipient_id',
    right_on='user_id',
    how='left'
)

df.columns

# 3. Compute risk_exposure_index exactly as established (llm_generated_prob * historical_fail_rate * security_clearance_level).
df['risk_exposure_index'] = (
    df['llm_generated_prob'] * df['historical_fail_rate'] * df['security_clearance_level']
).round(3)

# 4. Compute the non-shifted 10-minute rolling velocity window grouped by sender_address named velocity_10m
# First need to make sure timestamp is datetime format
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Sort df based on sender_address and timestamp
df = df.sort_values(['sender_address', 'timestamp'])

# Set index of df to timestamp
df = df.set_index('timestamp')

# Compute new column on a rolling window
df['velocity_10m'] = (
    df.groupby('sender_address')['email_id']
      .rolling('10min')
      .count()
      .reset_index(level=0, drop=True)
      .astype(int)
)

# Phase 3
# 1. Just the math alone will break the feature. Since he calculates for one email every 11 minutes, after 10 iterations the difference is already 10 minutes and the metrics won't align at all. 
# I would use feature stores to make sure all features are consistent and monitored for version control.

# 2. I would limit the number of incoming requests, based on a realistic number. Knowing that these types of attacks can occur, this would prevent a saturated cache memory.

df


,email_id,sender_address,recipient_id,reported_malicious,llm_generated_prob,domain_name,domain_age_days,historical_reputation_score,user_id,department,security_clearance_level,historical_fail_rate,risk_exposure_index,velocity_10m
timestamp,,,,,,,,,,,,,,
2026-05-18 00:02:54,m_3895,billing@invoice-vendor.com,u147,0,0.199779,invoice-vendor.com,120,0.85,u147,Executive Suite,2,0.086026,0.034,1
2026-05-18 00:03:27,m_2150,billing@invoice-vendor.com,u35,0,0.112534,invoice-vendor.com,120,0.85,u35,Sales,1,0.082731,0.009,2
2026-05-18 00:08:06,m_8887,billing@invoice-vendor.com,u3,0,0.044258,invoice-vendor.com,120,0.85,u3,Sales,3,0.464733,0.062,3
2026-05-18 00:09:19,m_9616,billing@invoice-vendor.com,u196,0,0.258976,invoice-vendor.com,120,0.85,u196,Human Resources,2,0.029083,0.015,4
2026-05-18 00:13:54,m_8225,billing@invoice-vendor.com,u24,0,0.085340,invoice-vendor.com,120,0.85,u24,Engineering,1,0.127579,0.011,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-19 23:21:32,m_3222,payroll@workday-benefits.co,u171,0,0.262631,workday-benefits.co,4,0.07,u171,Sales,1,0.036464,0.010,1
2026-05-19 23:26:37,m_6352,payroll@workday-benefits.co,u14,0,0.142177,workday-benefits.co,4,0.07,u14,Engineering,2,0.162013,0.046,2
2026-05-19 23:29:16,m_5836,payroll@workday-benefits.co,u142,0,0.126527,workday-benefits.co,4,0.07,u142,Sales,2,0.221538,0.056,3


In [80]:
# Phase 4
# Reset index made for rolling window feature engineering
df = df.reset_index()

# Identify columns for training and target
X_cols = df.drop(columns=['email_id', 'sender_address', 'recipient_id', 'domain_name', 'reported_malicious', 'timestamp'])
y_col = df['reported_malicious']

# Use time-based split for training and testing data
time_split = df['timestamp'].quantile(0.8)

# Split data into training and testing data
X_train = X_cols[df['timestamp'] < time_split]
X_test = X_cols[df['timestamp'] >= time_split]
y_train = y_col[df['timestamp'] < time_split]
y_test = y_col[df['timestamp'] >= time_split]

X_cols.columns

Index(['llm_generated_prob', 'domain_age_days', 'historical_reputation_score',
       'user_id', 'department', 'security_clearance_level',
       'historical_fail_rate', 'risk_exposure_index', 'velocity_10m'],
      dtype='object')

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score
)

# Specify categorical and numerical cols
categorical_cols = [
    'user_id',
    'department'
]

numerical_cols = [
    'llm_generated_prob',
    'domain_age_days',
    'historical_reputation_score',
    'security_clearance_level',
    'historical_fail_rate',
    'risk_exposure_index',
    'velocity_10m'
]

# Create preprocessor
preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numerical_cols)
    ]
)

# Create model using pipeline of preprocessors and the build model
model = Pipeline(
    steps=[
        ('preprocess', preprocess),
        ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))
    ]
)

# Fit model to training data
model.fit(X_train, y_train)

# Make target predictions using X_test data
y_preds = model.predict(X_test)

# Show the probabilities for the y_test set
y_prob = model.predict_proba(X=X_test)

# Evaluate model precision and recall
print(classification_report(y_pred=y_preds, y_true=y_test))
print(confusion_matrix(y_true=y_test, y_pred=y_preds))
print(precision_score(y_pred=y_preds, y_true=y_test))
print(recall_score(y_pred=y_preds, y_true=y_test))


# 4. Because accuracy is not the correct metric to use on a heavily imbalanced dataset. 
# The best metric to use is Precision and Recall, or F1-score. The PR-AUC will also help a great deal.

# Phase 5
# 1. It is clear that the recall should be as high as possible - leading to the decision threshold to be lower than the default value of 0.5 - something like 0.3 or 0.4
# depending on the capacity of the company to handle false positives. A false negative is thus detrimental.

# Phase 6
# 1. If an attacker knows this and manages to have the model train on faulty data, they would be able to slip through unnoticed.
# This could be by means of bypassing the network and creating false data.

# 2. SHAP identifies a feature importance list based on a specific prediction to be made. If an email to a VP was blocked,
# this would mean that the email was blocked due to some characteristics being high on the feature importance list. This is in essence the
# inner working of our classification model as it needs to decide whether an email is bad or good.

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1997
           1       1.00      1.00      1.00         3

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

[[1997    0]
 [   0    3]]
1.0
1.0
